# Lab 21: Calibrated scoring versus an LLM asked for a probability

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/prekshapraveen/Documents/lantern-lab2-submission.nosync | LLM mode: replay


In [2]:
# From the slide: Calibrating with scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import (
    CalibratedClassifierCV, calibration_curve)
from sklearn.metrics import brier_score_loss, roc_auc_score
log = pd.read_csv("data/reference/deal_log.csv")
FEATURES = ["revenue_usd_m", "growth", "ebitda_margin",
            "recurring_share", "top_customer_share",
            "asking_ev_ebitda"]
X_train, X_test, y_train, y_test, _, p_true_test = \
    train_test_split(log[FEATURES], log["advanced_to_ic"],
        log["p_true_advanced_to_ic"], test_size=0.3,
        random_state=7, stratify=log["advanced_to_ic"])
clf = CalibratedClassifierCV(
    estimator=LogisticRegression(max_iter=1000),
    method="isotonic", cv=5).fit(X_train, y_train)
p = clf.predict_proba(X_test)[:, 1]
print("AUC  ", roc_auc_score(y_test, p))
print("Brier", brier_score_loss(y_test, p))
print("Brier of the TRUE probs",
      brier_score_loss(y_test, p_true_test))  # floor
frac_pos, mean_pred = calibration_curve(
    y_test, p, n_bins=10)                   # the diagram

AUC   0.681842672413793
Brier 0.20523256118292932
Brier of the TRUE probs 0.18106188622222225


In [3]:
import numpy as np, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss, roc_auc_score
log = sd.deal_log(); ids = list(X_test.index)        # the identical test set for both approaches
rows = [{"deal_id": log.loc[i, "deal_id"], **{c: float(X_test.loc[i, c]) for c in X_test.columns}} for i in ids]
llm_scores = {s.deal_id: s.p for s in llm.structured(schemas.Scores, prompts.scorer_messages(rows), llm.P["llm"]["models"]["large"]).scores}
p_llm = np.array([llm_scores[log.loc[i, "deal_id"]] for i in ids]); p_mod = clf.predict_proba(X_test.loc[ids])[:, 1]; yt = y_test.loc[ids]
for name, p in (("calibrated model", p_mod), ("LLM-as-scorer", p_llm)):
    print(f"{name:16s} Brier {brier_score_loss(yt, p):.3f}  AUC {roc_auc_score(yt, p):.3f}  distinct values {len(set(np.round(p, 3)))}")
Path("reports/plots").mkdir(parents=True, exist_ok=True)
plt.plot(mean_pred, frac_pos, "o-", label="calibrated model"); plt.plot([0, 1], [0, 1], "--", label="perfect")
plt.xlabel("predicted"); plt.ylabel("observed"); plt.legend(); plt.savefig("reports/plots/reliability.png", dpi=120); print("saved reports/plots/reliability.png")

calibrated model Brier 0.205  AUC 0.682  distinct values 56
LLM-as-scorer    Brier 0.211  AUC 0.721  distinct values 11
saved reports/plots/reliability.png


Brier score rewards **calibration** (a stated 0.3 should come true about 30% of the time); AUC rewards only **ranking**. An approach can rank deals reasonably and still state badly calibrated probabilities.

**Exercise 21.** The LLM's probabilities cluster on round numbers. What does that do to its reliability diagram? Which approach should supply *probabilities* to the agent in Case Study 3, and would your answer change if the agent only needed a ranking?

In [4]:
from collections import Counter
from sklearn.calibration import calibration_curve   # in case it isn't imported above
print(sorted(Counter(float(x) for x in np.round(p_llm, 2)).items()))
fp, mp = calibration_curve(yt, p_llm, n_bins=10)
plt.plot(mp, fp, "s-", label="LLM-as-scorer"); plt.legend()
plt.savefig("reports/plots/reliability.png", dpi=120)
rng, yv, d = np.random.default_rng(7), yt.values, []
for _ in range(2000):
    i = rng.integers(0, len(yv), len(yv))
    if yv[i].min() == yv[i].max(): continue
    d.append(roc_auc_score(yv[i], p_llm[i]) - roc_auc_score(yv[i], p_mod[i]))
print(np.percentile(d, [2.5, 97.5]).round(3))

[(0.05, 10), (0.1, 3), (0.2, 8), (0.3, 13), (0.4, 8), (0.5, 16), (0.6, 13), (0.7, 9), (0.8, 3), (0.9, 3), (0.95, 4)]
[-0.111  0.209]


### Exercise 21: answer

**How often did the LLM use each value?** Only 11 distinct values across the 90 test deals, all
round numbers: 0.05 (10 deals), 0.1 (3), 0.2 (8), 0.3 (13), 0.4 (8), 0.5 (16), 0.6 (13), 0.7 (9),
0.8 (3), 0.9 (3), 0.95 (4). It never says 0.27 or 0.63: it answers on a coarse, rounded scale, and
0.5 ("no idea") is its most common answer.

**What does that do to a reliability diagram?** The LLM's points can only sit at those 11 values, so
its curve is a few large clumps rather than a smooth line, and each clump mixes deals with very
different true probabilities. Its Brier score (0.211) is worse than the calibrated model's (0.205):
its probabilities are less trustworthy as probabilities. (The LLM curve is added to
`reports/plots/reliability.png`.)

**Does the LLM rank deals better?** Its AUC is higher (0.721 vs 0.682), but the bootstrap 95% interval
for the difference (LLM − model) is **[−0.111, 0.209]**, which includes zero. So we can't claim the LLM
ranks deals better; the gap could be chance on 90 deals.

**Who supplies probabilities to the agent?** The calibrated model: lower Brier, and its calibration can
be tested and repaired. If the agent only needed a ranking, the LLM still isn't demonstrably better. The
LLM's judgement could still be useful as an *input feature* to the calibrated model, not as the
probability itself.